In [ ]:
!pip install datasets requests torch

In [ ]:
import time
import random
import statistics
import requests
import torch

from datasets import load_dataset

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

reporting = load_dataset("JayShah07/reporting_final_dataset")

print("Dataset structure:")
print(reporting)

print("\nDataset sizes:")
print(f"Train: {len(reporting['train'])}")
print(f"Validation: {len(reporting['validation'])}")
print(f"Test: {len(reporting['test'])}")

Using device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/47.8k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/12.1k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/12.3k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3097 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/387 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/388 [00:00<?, ? examples/s]

Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['query', 'holdings', 'capital_gains', 'scheme_wise_returns', 'investment_account_wise_returns', 'portfolio_update', 'Current Year', 'Previous Year', 'Daily', 'Monthly', 'Weekly', 'Yearly', 'None_date', 'None_module', 'labels'],
        num_rows: 3097
    })
    validation: Dataset({
        features: ['query', 'holdings', 'capital_gains', 'scheme_wise_returns', 'investment_account_wise_returns', 'portfolio_update', 'Current Year', 'Previous Year', 'Daily', 'Monthly', 'Weekly', 'Yearly', 'None_date', 'None_module', 'labels'],
        num_rows: 387
    })
    test: Dataset({
        features: ['query', 'holdings', 'capital_gains', 'scheme_wise_returns', 'investment_account_wise_returns', 'portfolio_update', 'Current Year', 'Previous Year', 'Daily', 'Monthly', 'Weekly', 'Yearly', 'None_date', 'None_module', 'labels'],
        num_rows: 388
    })
})

Dataset sizes:
Train: 3097
Validation: 387
Test: 388


In [ ]:
TEST_SAMPLES = 130

test_queries = [
    example["query"]
    for example in reporting["test"].select(range(TEST_SAMPLES))
]

print(f"Loaded {len(test_queries)} test queries")
print("\nSample queries:")
for q in test_queries[:5]:
    print("-", q)


Loaded 130 test queries

Sample queries:
- Share my holdings report
- Give my returns divided by investment accounts Provide account-wise investment returns
- Show how my portfolio is performing currently as an annual report
- Show my gains for the selected period Provide my capital gains statement
- Can you generate a scheme wise report for my investments for the current year?


In [ ]:
API_URL = "https://classification-model-mlops.onrender.com/api/predict"

HEADERS = {
    "accept": "application/json",
    "Content-Type": "application/json"
}


In [ ]:
latencies = []
success = 0
failures = 0

print("🚀 Starting online inference using TEST set...\n")

for idx, query in enumerate(test_queries, start=1):
    payload = {"text": query}
    start_time = time.time()

    try:
        response = requests.post(
            API_URL,
            json=payload,
            headers=HEADERS,
            timeout=10
        )

        latency = time.time() - start_time
        latencies.append(latency)

        if response.status_code == 200:
            success += 1
            result = response.json()

            print(
                f"[{idx:03}] ✅ "
                f"Latency={latency:.3f}s | "
                f"Module={result['module_best']} | "
                f"Date={result['date_best']}"
            )
        else:
            failures += 1
            print(f"[{idx:03}] ❌ HTTP {response.status_code}")

    except Exception as e:
        failures += 1
        print(f"[{idx:03}] ❌ Exception: {e}")

    # Simulate real user traffic
    time.sleep(random.uniform(0.2, 0.6))


🚀 Starting online inference using TEST set...

[001] ✅ Latency=0.923s | Module=holdings | Date=None_date
[002] ✅ Latency=0.981s | Module=investment_account_wise_returns | Date=None_date
[003] ✅ Latency=0.619s | Module=portfolio_update | Date=Yearly
[004] ✅ Latency=0.398s | Module=capital_gains | Date=None_date
[005] ✅ Latency=0.850s | Module=scheme_wise_returns | Date=None_date
[006] ✅ Latency=0.998s | Module=None_module | Date=None_date
[007] ✅ Latency=0.423s | Module=portfolio_update | Date=Yearly
[008] ✅ Latency=1.279s | Module=holdings | Date=None_date
[009] ✅ Latency=0.196s | Module=capital_gains | Date=Previous Year
[010] ✅ Latency=0.748s | Module=portfolio_update | Date=Monthly
[011] ✅ Latency=0.435s | Module=capital_gains | Date=Current Year
[012] ✅ Latency=1.239s | Module=investment_account_wise_returns | Date=None_date
[013] ✅ Latency=0.445s | Module=None_module | Date=None_date
[014] ✅ Latency=0.689s | Module=scheme_wise_returns | Date=Current Year
[015] ✅ Latency=0.563s | M

In [ ]:
print("\n================ SUMMARY ================")
print(f"Total requests   : {len(test_queries)}")
print(f"Successful calls : {success}")
print(f"Failures         : {failures}")

if latencies:
    print(f"Average latency  : {statistics.mean(latencies):.3f}s")
    print(f"P95 latency      : {statistics.quantiles(latencies, n=20)[18]:.3f}s")
    print(f"Max latency      : {max(latencies):.3f}s")



================ SUMMARY ================
Total requests   : 130
Successful calls : 130
Failures         : 0
Average latency  : 0.703s
P95 latency      : 1.158s
Max latency      : 1.279s
